# Beispiele: Neuronale Netze

Ein Neuron von Hand, Backpropagation mit Gradient-Check, dann ein Netz mit scikit-learn (`MLPClassifier`) auf zwei Halbmonden: Vergleich mit Baseline und logistischer Regression, Grösse des Netzes, learning rate, L2 und Early Stopping, Loss-Kurven. Keras und TensorFlow sind bewusst nicht nötig: Die Ideen sind dieselben, die Installation ist es nicht.

- **Herkunft:** selbst erstellt für die Lernapp, kein HSLU-Kursmaterial. Die Daten werden erzeugt.
- **Ausführen:** von oben nach unten, das Notebook hängt von keinem anderen ab. Lokal in der Lernapp oder auf jedem anderen PC im Browser: [In Colab öffnen](https://colab.research.google.com/github/rotscher-baumi/lernapp-cas-ml/blob/main/notebooks/beispiele/08_Neuronale_Netze.ipynb)
- **Zufall:** `SEED = 42`. Die Startgewichte sind zufällig, darum ändern sich Zahlen mit einem anderen seed leicht. Vergleiche nur deutliche Unterschiede.

In [ ]:
import warnings

import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_moons
from sklearn.dummy import DummyClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=ConvergenceWarning)  # wir begrenzen max_iter absichtlich

SEED = 42
np.set_printoptions(precision=4, suppress=True)

## 1 · Ein Trainingsschritt von Hand

Netz mit 2 Eingängen, 2 hidden Neuronen (Sigmoid) und 1 Ausgang (Sigmoid), Loss = Binary Cross-Entropy. Das ist dasselbe Netz wie in der Demo «Ein Trainingsschritt» der Lernapp. Wir rechnen Forward pass, Loss und Backpropagation in NumPy und prüfen die Gradienten gegen **numerische Gradienten** (Differenzenquotient). Stimmen beide überein, ist die Kettenregel richtig umgesetzt.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


x = np.array([1.0, 0.5])
y = 1.0
W1 = np.array([[0.5, -0.4], [0.3, 0.8]])   # (hidden, input)
b1 = np.array([0.1, -0.1])
W2 = np.array([0.7, -0.6])
b2 = 0.2


def forward(W1, b1, W2, b2):
    z1 = W1 @ x + b1
    a1 = sigmoid(z1)
    z2 = W2 @ a1 + b2
    return a1, sigmoid(z2)


def loss_fn(W1, b1, W2, b2):
    _, p = forward(W1, b1, W2, b2)
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))


a1, p = forward(W1, b1, W2, b2)
print("Vorhersage ŷ =", round(float(p), 4), " Loss =", round(float(loss_fn(W1, b1, W2, b2)), 4))

In [ ]:
# Backpropagation: bei Sigmoid-Ausgang und Cross-Entropy ist der Fehler am Ausgang einfach ŷ - y
d2 = p - y
gW2 = d2 * a1
gb2 = d2
d1 = (W2 * d2) * a1 * (1 - a1)          # Fehler zurück auf die hidden layer
gW1 = np.outer(d1, x)
gb1 = d1


def numeric_grad(f, params, h=1e-6):
    """Differenzenquotient für jeden Eintrag jedes Parameter-Arrays."""
    grads = []
    for P in params:
        G = np.zeros_like(P, dtype=float)
        for idx in np.ndindex(P.shape):
            old = P[idx]
            P[idx] = old + h
            up = f(*params)
            P[idx] = old - h
            down = f(*params)
            P[idx] = old
            G[idx] = (up - down) / (2 * h)
        grads.append(G)
    return grads


params = [W1.copy(), b1.copy(), W2.copy(), np.array([b2])]
num = numeric_grad(lambda a, b, c, d: loss_fn(a, b, c, d[0]), params)
ana = [gW1, gb1, gW2, np.array([gb2])]
for name, a, n in zip(["W1", "b1", "W2", "b2"], ana, num):
    print(name, "grösster Unterschied:", float(np.max(np.abs(a - n))))
assert all(np.allclose(a, n, atol=1e-7) for a, n in zip(ana, num))

Jetzt mehrere Update-Schritte `w ← w − α · g`. Die Loss sinkt, die Vorhersage nähert sich `y = 1`.

In [ ]:
lr = 0.5
P = [W1.copy(), b1.copy(), W2.copy(), float(b2)]
history = []
for step in range(30):
    W1_, b1_, W2_, b2_ = P
    a1_, p_ = forward(W1_, b1_, W2_, b2_)
    history.append(float(loss_fn(W1_, b1_, W2_, b2_)))
    d2_ = p_ - y
    d1_ = (W2_ * d2_) * a1_ * (1 - a1_)
    P = [W1_ - lr * np.outer(d1_, x), b1_ - lr * d1_, W2_ - lr * d2_ * a1_, b2_ - lr * d2_]

print("Loss vorher:", round(history[0], 4), " nach 30 Schritten:", round(history[-1], 4))
plt.figure(figsize=(4, 2.5))
plt.plot(history)
plt.xlabel("Update-Schritt")
plt.ylabel("Loss")
plt.title("Ein Beispiel, α = 0.5")
plt.tight_layout()

## 2 · Daten, Split und Baselines

Zwei verschränkte Halbmonde mit Rauschen: nicht linear trennbar. Aufteilung in **train**, **validation** und **test**. Der Scaler lernt nur auf den Trainingsdaten. Das Ergebnis jedes Modells steht neben einer Baseline (immer die häufigste Klasse) und der logistischen Regression (ein einzelnes Neuron).

In [ ]:
X, y_all = make_moons(n_samples=600, noise=0.3, random_state=SEED)
X_tmp, X_test, y_tmp, y_test = train_test_split(X, y_all, test_size=0.2, stratify=y_all, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=0.25, stratify=y_tmp, random_state=SEED)

scaler = StandardScaler().fit(X_train)            # nur auf train gelernt
X_train, X_val, X_test = (scaler.transform(a) for a in (X_train, X_val, X_test))
print("train / validation / test:", len(X_train), len(X_val), len(X_test))

base = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
logit = LogisticRegression().fit(X_train, y_train)
print("Accuracy Validation, Baseline:             ", round(base.score(X_val, y_val), 3), f"(n = {len(X_val)})")
print("Accuracy Validation, logistische Regression:", round(logit.score(X_val, y_val), 3))

## 3 · Ein Netz: wie viele Neuronen?

`MLPClassifier` mit einer hidden layer, ReLU, Adam. Wir vergleichen die Grösse der hidden layer auf der **Validation**. Weil die Startgewichte zufällig sind, mitteln wir über fünf seeds.

In [ ]:
def fit_mlp(hidden=(32,), seed=SEED, **kw):
    params = dict(hidden_layer_sizes=hidden, activation="relu", solver="adam", max_iter=500, random_state=seed)
    params.update(kw)
    return MLPClassifier(**params).fit(X_train, y_train)


sizes = [1, 2, 4, 8, 32, 128]
print("Neuronen | Parameter | Accuracy train | Accuracy validation (Mittel über 5 seeds)")
for h in sizes:
    tr, va = [], []
    for s in range(5):
        m = fit_mlp((h,), seed=SEED + s)
        tr.append(m.score(X_train, y_train))
        va.append(m.score(X_val, y_val))
    n_par = (2 + 1) * h + (h + 1) * 1
    print(f"{h:8d} | {n_par:9d} | {np.mean(tr):14.3f} | {np.mean(va):.3f}")

Lesen: Mit 1 oder 2 Neuronen reicht die Kapazität nicht, das Netz ist sogar schlechter als die logistische Regression. Ab etwa 4 Neuronen holt es sie ein, mit mehr Neuronen kommt auf der Validation nur noch wenig dazu, auf train dagegen mehr. Der Abstand zwischen train und validation wächst mit der Grösse.

In [ ]:
def boundary(model, ax, title):
    xx, yy = np.meshgrid(np.linspace(-2.5, 2.5, 200), np.linspace(-2.5, 2.5, 200))
    zz = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, zz, alpha=0.25, cmap="coolwarm")
    ax.scatter(*X_train.T, c=y_train, cmap="coolwarm", edgecolor="k", s=18)
    ax.set_title(title, fontsize=9)


fig, axs = plt.subplots(1, 3, figsize=(11, 3.2))
for ax, (name, m) in zip(axs, [("logistische Regression", logit), ("MLP, 4 Neuronen", fit_mlp((4,))), ("MLP, 128 Neuronen", fit_mlp((128,)))]):
    boundary(m, ax, f"{name}: Validation {m.score(X_val, y_val):.3f}")
plt.tight_layout()

## 4 · Loss-Kurven lesen und Early Stopping

`loss_curve_` ist die Training-Loss je epoch. Für die Validation-Loss trainieren wir mit `partial_fit` Epoche für Epoche und messen selbst. Das Minimum der Validation-Loss ist der Punkt für Early Stopping.

In [ ]:
from sklearn.metrics import log_loss


def curves(hidden, epochs=300, seed=SEED, **kw):
    m = MLPClassifier(hidden_layer_sizes=hidden, activation="relu", solver="adam", batch_size=32,
                      random_state=seed, **kw)
    tr, va = [], []
    for _ in range(epochs):
        m.partial_fit(X_train, y_train, classes=[0, 1])
        tr.append(log_loss(y_train, m.predict_proba(X_train)))
        va.append(log_loss(y_val, m.predict_proba(X_val)))
    return tr, va


fig, axs = plt.subplots(1, 2, figsize=(10, 3))
for ax, (hidden, name) in zip(axs, [((4,), "klein: 4 Neuronen"), ((128, 128), "gross: 2 × 128 Neuronen")]):
    tr, va = curves(hidden)
    best = int(np.argmin(va))
    ax.plot(tr, label="train")
    ax.plot(va, label="validation")
    ax.axvline(best, color="k", ls=":", label=f"Minimum bei epoch {best + 1}")
    ax.set_title(f"{name}: min. Validation-Loss {min(va):.3f}", fontsize=9)
    ax.set_xlabel("epoch")
    ax.set_ylabel("Loss")
    ax.legend(fontsize=8)
plt.tight_layout()

Das grosse Netz zeigt typisches overfitting: Die Training-Loss sinkt weiter, die Validation-Loss steigt nach ihrem Minimum wieder. **Early Stopping** hört beim Minimum auf. In scikit-learn geht das mit `early_stopping=True`: Es hält intern 10 % der Trainingsdaten als Validation zurück und stoppt, wenn sich der Score `n_iter_no_change` epochs lang nicht verbessert. Bei unseren 360 Trainingspunkten sind das nur 36 Punkte, der Stopp-Zeitpunkt ist also verrauscht. Vergleiche darum die Loss auf der echten Validation, nicht nur die Accuracy.

In [ ]:
plain = fit_mlp((128, 128), max_iter=300, batch_size=32)
early = fit_mlp((128, 128), max_iter=300, batch_size=32, early_stopping=True, n_iter_no_change=20)
for name, m in [("ohne Early Stopping", plain), ("mit  Early Stopping", early)]:
    print(f"{name}: epochs = {m.n_iter_:3d}  Accuracy validation = {m.score(X_val, y_val):.3f}  Loss validation = {log_loss(y_val, m.predict_proba(X_val)):.3f}")
print("Baseline:", round(base.score(X_val, y_val), 3), " logistische Regression:", round(logit.score(X_val, y_val), 3))

## 5 · Regularisierung mit L2 (`alpha`) und die learning rate

`alpha` ist der L2-Strafterm (weight decay) wie bei Ridge. Er glättet die Grenze. Die **learning rate** (`learning_rate_init`) ist der wichtigste Regler des Optimizers: zu klein lernt nichts in der Zeit, zu gross springt.

In [ ]:
print("alpha (L2) | Accuracy train | Accuracy validation")
for a in [0.0001, 0.01, 0.1, 1.0, 3.0]:
    m = fit_mlp((128,), alpha=a)
    print(f"{a:11g} | {m.score(X_train, y_train):14.3f} | {m.score(X_val, y_val):.3f}")

print()
print("learning rate | epochs bis Ende | Accuracy validation | letzte Training-Loss")
for lr in [0.0001, 0.001, 0.01, 0.1, 1.0]:
    m = fit_mlp((32,), learning_rate_init=lr, max_iter=200)
    print(f"{lr:13g} | {m.n_iter_:15d} | {m.score(X_val, y_val):19.3f} | {m.loss_curve_[-1]:.3f}")

## 6 · Optimizer im Vergleich

SGD, SGD mit Momentum und Adam auf demselben Netz und denselben Daten, mit fester Zahl epochs.

In [ ]:
def run(solver, **kw):
    m = MLPClassifier(hidden_layer_sizes=(32,), activation="relu", solver=solver, batch_size=32,
                      max_iter=60, random_state=SEED, **kw).fit(X_train, y_train)
    return m


configs = {
    "SGD": dict(solver="sgd", momentum=0.0, learning_rate_init=0.01),
    "SGD + Momentum 0.9": dict(solver="sgd", momentum=0.9, nesterovs_momentum=False, learning_rate_init=0.01),
    "Adam": dict(solver="adam", learning_rate_init=0.001),
}
plt.figure(figsize=(5.5, 3))
for name, cfg in configs.items():
    m = run(**cfg)
    plt.plot(m.loss_curve_, label=f"{name} (Validation {m.score(X_val, y_val):.3f})")
plt.xlabel("epoch")
plt.ylabel("Training-Loss")
plt.legend(fontsize=8)
plt.tight_layout()

## 7 · Testen, genau einmal

Architektur und Hyperparameter stehen fest (32 Neuronen, alpha = 0.01, aus der Validation gewählt). Jetzt das Testset, einmal. Danach nichts mehr am Setup ändern.

In [ ]:
final = fit_mlp((32,), alpha=0.01)
print(f"Test, final (MLP 32 Neuronen):  {final.score(X_test, y_test):.3f}  (n = {len(X_test)})")
print(f"Test, logistische Regression:   {logit.score(X_test, y_test):.3f}")
print(f"Test, Baseline:                 {base.score(X_test, y_test):.3f}")

## Zum Weiterdenken

- Warum lernt `MLPClassifier(hidden_layer_sizes=(1,))` die Monde nicht, ein Netz mit 8 Neuronen aber schon?
- Was ändert sich, wenn du die Eingaben nicht skalierst?
- Setze `activation="identity"` (linear): Wie sieht die Grenze aus, und warum?
- Wiederhole Abschnitt 3 mit `SEED + 100`. Welche Unterschiede bleiben, welche verschwinden?